# OCR 검증 파이프라인: Load -> OCR -> Validate

cron 수집 이후 저장된 PDF를 입력으로 사용합니다. 원본 PDF를 수정하지 않고 엔진별 원문, 공통 정규화 결과, 검증 대기열을 분리 저장합니다.

In [1]:
from __future__ import annotations

import base64
import hashlib
import html
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import fitz
import pandas as pd
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
if (cwd / 'data').is_dir() and (cwd / 'notebooks').is_dir():
    BACKEND_ROOT = cwd
elif (cwd.parent / 'data').is_dir() and cwd.name == 'notebooks':
    BACKEND_ROOT = cwd.parent
else:
    raise RuntimeError(f'프로젝트 루트 또는 notebooks 폴더를 찾을 수 없습니다: {cwd}')

RAW_PDF_DIR = BACKEND_ROOT / 'data/documents/raw'
VISION_DIR = BACKEND_ROOT / 'data/documents/vision'
OUTPUT_ROOT = BACKEND_ROOT / 'notebooks/data/06_ocr_validation_pipeline'
RAW_OUTPUT = OUTPUT_ROOT / 'raw'
NORMALIZED_OUTPUT = OUTPUT_ROOT / 'normalized'
VALIDATION_OUTPUT = OUTPUT_ROOT / 'validation'
for path in (RAW_OUTPUT, NORMALIZED_OUTPUT, VALIDATION_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

load_dotenv(BACKEND_ROOT / '.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
UPSTAGE_API_KEY = os.getenv('UPSTAGE_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# 기본값은 외부 전송 없이 기존 Vision 결과를 읽고 검증만 수행합니다.
RUN_VISION = True
RUN_UPSTAGE = True
RUN_CLAUDE = False
ACTIVE_ENGINES = {'vision'} | ({'upstage'} if RUN_UPSTAGE else set()) | ({'claude'} if RUN_CLAUDE else set())
MIN_COMPARISON_ENGINES = 2
OPENAI_VISION_MODEL = os.getenv('OPENAI_VISION_MODEL', 'gpt-5.4-mini-2026-03-17')
CLAUDE_VISION_MODEL = os.getenv('CLAUDE_VISION_MODEL', 'claude-sonnet-4-6')

# None이면 raw 아래의 모든 PDF를 대상으로 합니다. 개발 중에는 예: {'shinhan'}처럼 제한합니다.
TARGET_ISSUERS: set[str] | None = {'BC', 'NH'}


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_documents() -> list[dict]:
    documents = []
    for pdf_path in sorted(RAW_PDF_DIR.glob('*/*.pdf')):
        issuer = pdf_path.parent.name
        if TARGET_ISSUERS is not None and issuer not in TARGET_ISSUERS:
            continue
        with fitz.open(pdf_path) as pdf:
            page_count = len(pdf)
        documents.append({
            'document_id': sha256_file(pdf_path),
            'issuer': issuer,
            'file_name': pdf_path.name,
            'pdf_path': str(pdf_path),
            'page_count': page_count,
            'loaded_at': datetime.now(timezone.utc).isoformat(),
        })
    return documents

documents = load_documents()
manifest_path = OUTPUT_ROOT / 'document_manifest.json'
manifest_path.write_text(json.dumps(documents, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'로드한 PDF: {len(documents)}개')
display(pd.DataFrame(documents)[['issuer', 'file_name', 'page_count']].head())


로드한 PDF: 20개


,issuer,file_name,page_count
0,BC,BC_Baro_Clear_Plus.pdf,2
1,BC,BC_Baro_KaPick.pdf,2
2,BC,BC_BizCorporate.pdf,2
3,BC,BC_Biz_AirMoney.pdf,2
4,BC,BC_Business_Sky.pdf,4


In [3]:
PAGE_MARKER = re.compile(r'^\[PAGE (\d+)\]\s*$', re.M)

def require_env(name: str, value: str | None) -> str:
    if not value:
        raise RuntimeError(f'{name} 환경변수가 없습니다. 프로젝트 루트의 .env에만 설정하세요.')
    return value

def document_key(document: dict) -> str:
    return f"{document['issuer']}__{Path(document['file_name']).stem}"

def render_page(document: dict, page_number: int) -> bytes:
    with fitz.open(document['pdf_path']) as pdf:
        pixmap = pdf[page_number - 1].get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
        return pixmap.tobytes('png')

def split_vision_pages(text: str) -> dict[int, str]:
    markers = list(PAGE_MARKER.finditer(text))
    pages = {}
    for index, marker in enumerate(markers):
        end = markers[index + 1].start() if index + 1 < len(markers) else len(text)
        pages[int(marker.group(1))] = text[marker.end():end].strip()
    return pages

def read_existing_vision(document: dict) -> list[dict]:
    path = VISION_DIR / document['issuer'] / f"{Path(document['file_name']).stem}.txt"
    if not path.exists():
        return []
    return [{'engine': 'vision', 'page_number': page, 'page_text': text, 'source': str(path)}
            for page, text in split_vision_pages(path.read_text(encoding='utf-8')).items()]

def run_openai_vision(document: dict, page_number: int) -> str:
    from openai import OpenAI
    image = base64.b64encode(render_page(document, page_number)).decode('ascii')
    response = OpenAI(api_key=require_env('OPENAI_API_KEY', OPENAI_API_KEY)).responses.create(
        model=OPENAI_VISION_MODEL,
        input=[{'role': 'user', 'content': [
            {'type': 'input_text', 'text': '카드 안내 PDF 페이지의 모든 텍스트를 읽기 순서대로 전사하세요. 표는 Markdown 표로 작성하고, 요약이나 추론은 하지 마세요.'},
            {'type': 'input_image', 'image_url': f'data:image/png;base64,{image}', 'detail': 'high'},
        ]}],
    )
    return response.output_text.strip()

def run_upstage(document: dict) -> dict:
    import requests
    with Path(document['pdf_path']).open('rb') as stream:
        response = requests.post('https://api.upstage.ai/v1/document-digitization',
            headers={'Authorization': f"Bearer {require_env('UPSTAGE_API_KEY', UPSTAGE_API_KEY)}"},
            files={'document': stream}, data={'model': 'document-parse', 'ocr': 'force'}, timeout=180)
    response.raise_for_status()
    return response.json()

def run_claude(document: dict, page_number: int) -> str:
    import requests
    image = base64.b64encode(render_page(document, page_number)).decode('ascii')
    response = requests.post('https://api.anthropic.com/v1/messages',
        headers={'x-api-key': require_env('ANTHROPIC_API_KEY', ANTHROPIC_API_KEY), 'anthropic-version': '2023-06-01', 'content-type': 'application/json'},
        json={'model': CLAUDE_VISION_MODEL, 'max_tokens': 8192, 'temperature': 0, 'messages': [{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': image}},
            {'type': 'text', 'text': '카드 안내 PDF 페이지의 모든 텍스트를 읽기 순서대로 Markdown으로 전사하세요. 표는 Markdown 표로 작성하고, 요약이나 추론은 하지 마세요.'},
        ]}]}, timeout=180)
    if not response.ok:
        raise RuntimeError(f'Claude 요청 실패 ({response.status_code}): {response.text[:1000]}')
    return '\n'.join(block.get('text', '') for block in response.json().get('content', []) if block.get('type') == 'text')


In [4]:
def normalize_text(text: str) -> str:
    text = html.unescape(text or '')
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.I)
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

FACT_PATTERNS = {
    'percent': re.compile(r'\d+(?:\.\d+)?\s*%'),
    'money': re.compile(r'\d{1,3}(?:,\d{3})*\s*(?:원|만원|천원)'),
    'previous_month_spend': re.compile(r'전월\s*(?:실적|이용금액)[^\n]{0,40}'),
}

def extract_facts(page_text: str) -> dict[str, list[str]]:
    normalized = normalize_text(page_text)
    return {kind: sorted(set(match.group(0).replace(' ', '') for match in pattern.finditer(normalized)))
            for kind, pattern in FACT_PATTERNS.items()}

def build_page_record(document: dict, engine: str, page_number: int, page_text: str, source: str) -> dict:
    return {
        'document_id': document['document_id'], 'issuer': document['issuer'],
        'file_name': document['file_name'], 'page_number': page_number,
        'engine': engine, 'source': source, 'normalized_text': normalize_text(page_text),
        'facts': extract_facts(page_text),
    }

# 이미 저장된 신규 OCR 원문은 재사용하므로, 중단 뒤 재실행해도 같은 페이지를 다시 호출하지 않습니다.
def cached_page(engine: str, document: dict, page_number: int) -> tuple[str | None, Path]:
    path = RAW_OUTPUT / engine / document_key(document) / f'{page_number:03d}.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8')).get('page_text'), path
    return None, path

def save_page(path: Path, model: str, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({'model': model, 'page_text': text}, ensure_ascii=False, indent=2), encoding='utf-8')

records, runtime_errors = [], []
for document in documents:
    if RUN_VISION:
        for page_number in range(1, document['page_count'] + 1):
            text, path = cached_page('vision', document, page_number)
            try:
                if text is None:
                    text = run_openai_vision(document, page_number)
                    save_page(path, OPENAI_VISION_MODEL, text)
                records.append(build_page_record(document, 'vision', page_number, text, str(path)))
            except Exception as exc:
                runtime_errors.append({'engine': 'vision', 'file_name': document['file_name'], 'page_number': page_number, 'error': str(exc)})
    else:
        for result in read_existing_vision(document):
            records.append(build_page_record(document, result['engine'], result['page_number'], result['page_text'], result['source']))
    if RUN_UPSTAGE:
        path = RAW_OUTPUT / 'upstage' / f"{document_key(document)}.json"
        try:
            raw = json.loads(path.read_text(encoding='utf-8')) if path.exists() else run_upstage(document)
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text(json.dumps(raw, ensure_ascii=False, indent=2), encoding='utf-8')
            for page_number in range(1, document['page_count'] + 1):
                parts = [e.get('content', {}).get('text') or e.get('content', {}).get('markdown') or e.get('content', {}).get('html') or '' for e in raw.get('elements', []) if e.get('page') == page_number]
                records.append(build_page_record(document, 'upstage', page_number, '\n'.join(parts), str(path)))
        except Exception as exc:
            runtime_errors.append({'engine': 'upstage', 'file_name': document['file_name'], 'page_number': None, 'error': str(exc)})
    if RUN_CLAUDE:
        for page_number in range(1, document['page_count'] + 1):
            text, path = cached_page('claude', document, page_number)
            try:
                if text is None:
                    text = run_claude(document, page_number)
                    save_page(path, CLAUDE_VISION_MODEL, text)
                records.append(build_page_record(document, 'claude', page_number, text, str(path)))
            except Exception as exc:
                runtime_errors.append({'engine': 'claude', 'file_name': document['file_name'], 'page_number': page_number, 'error': str(exc)})

NORMALIZED_OUTPUT.joinpath('page_records.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
VALIDATION_OUTPUT.joinpath('runtime_errors.json').write_text(json.dumps(runtime_errors, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'정규화된 페이지 레코드: {len(records)}개 / OCR 오류: {len(runtime_errors)}개')


정규화된 페이지 레코드: 92개


In [5]:
def validate_page(group: list[dict]) -> dict:
    engines = {row['engine'] for row in group}
    facts_by_engine = {row['engine']: row['facts'] for row in group}
    all_facts = {kind: {engine: set(facts.get(kind, [])) for engine, facts in facts_by_engine.items()} for kind in FACT_PATTERNS}
    mismatches = {kind: {engine: sorted(values) for engine, values in values_by_engine.items()}
                  for kind, values_by_engine in all_facts.items()
                  if len({tuple(sorted(values)) for values in values_by_engine.values()}) > 1}
    missing_engine = not ACTIVE_ENGINES.issubset(engines)
    insufficient_comparison = len(engines) < MIN_COMPARISON_ENGINES
    status = 'needs_review' if missing_engine or insufficient_comparison or mismatches else 'auto_candidate'
    base = group[0]
    return {
        'document_id': base['document_id'], 'issuer': base['issuer'], 'file_name': base['file_name'],
        'page_number': base['page_number'], 'engines_present': sorted(engines),
        'status': status, 'missing_engine': missing_engine, 'insufficient_comparison': insufficient_comparison, 'fact_mismatches': mismatches,
    }

groups = {}
for record in records:
    groups.setdefault((record['document_id'], record['page_number']), []).append(record)
validation_rows = [validate_page(group) for group in groups.values()]
VALIDATION_OUTPUT.joinpath('validation_results.json').write_text(json.dumps(validation_rows, ensure_ascii=False, indent=2), encoding='utf-8')
review_queue = pd.DataFrame([row for row in validation_rows if row['status'] == 'needs_review'])
review_queue.to_json(VALIDATION_OUTPUT / 'review_queue.json', orient='records', force_ascii=False, indent=2)
print('자동 통과 후보:', sum(row['status'] == 'auto_candidate' for row in validation_rows))
print('검토 대기:', len(review_queue))
display(review_queue.head())

# 운영 전제: auto_candidate도 초기에는 무작위 표본을 사람 검토하고, 그 결과로 임계값을 보정합니다.


자동 통과 후보: 0
검토 대기: 46


,document_id,issuer,file_name,page_number,engines_present,status,missing_engine,fact_mismatches
0,4ccb996117c9cf592d4110525a0084a0859caeffdb243b...,BC,BC_Baro_Clear_Plus.pdf,1,"[upstage, vision]",needs_review,True,{}
1,4ccb996117c9cf592d4110525a0084a0859caeffdb243b...,BC,BC_Baro_Clear_Plus.pdf,2,"[upstage, vision]",needs_review,True,{'previous_month_spend': {'vision': ['전월실적15만원...
2,523ab63a4c0511f8dccdfc020e684d43d15871d9b9d0c5...,BC,BC_Baro_KaPick.pdf,1,"[upstage, vision]",needs_review,True,{}
3,523ab63a4c0511f8dccdfc020e684d43d15871d9b9d0c5...,BC,BC_Baro_KaPick.pdf,2,"[upstage, vision]",needs_review,True,"{'previous_month_spend': {'vision': ['전월실적,한도없..."
4,49a9cada9ecbfc2fb551514463d78fd14be3dfc7d15f1a...,BC,BC_BizCorporate.pdf,1,"[upstage, vision]",needs_review,True,{'previous_month_spend': {'vision': ['전월실적미달시에...
